# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Refresh / Content Opportunity Scoring

**Task type:** This is a **ranking** problem (with a scoring component underneath it).
I'm not just classifying pages as "good" or "bad" — I need to order all pages by how
urgently they need review, so a reviewer can work through a ranked list starting from
the top. The underlying model produces a score per page (probability of decline), and
that score is what drives the ranking. So concretely: scoring → ranking, not plain
classification, because the output a reviewer actually uses is an ordered list, not a
single label per page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** My true target — "will this page's traffic meaningfully decline in
the near future" — isn't directly observable in advance, so I use a proxy: the
existing `trend_direction` column (down / flat / new / stable / up), specifically
treating `trend_direction == "down"` as the positive class the model learns to catch.

This is a reasonable proxy because it's based on real recent trend data already
computed from the page's traffic history, not a guess — but it's still a proxy, not
ground truth, because a page trending down over the observed window may recover on
its own, and a page currently flat could decline right after this snapshot. The proxy
captures "recently declining" as a stand-in for "needs review," which is a
reasonable but imperfect approximation of the real business question.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric:** Precision@50 — of the top 50 pages the model ranks highest for
review, what fraction are truly declining (trend_direction == "down")?

This fits the real decision better than overall accuracy would, because a reviewer
only has time to work through a limited list (roughly the top 50) in a review cycle —
accuracy across all 30,000 pages doesn't matter if the top of the ranked list is full
of false positives. Precision@50 directly measures whether the reviewer's limited time
gets spent on pages that actually need it.

From the Week 1 pipeline: the hand-written baseline rule scored 0.24 Precision@50,
while the random forest scored 0.74 — meaning the model gets roughly 3x more of the
top 50 flagged pages right.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/susheel123-sketch/Flyrank-Internship-ml"
REPO_DIR = "YOUR_REPO_NAME"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Unit of analysis: one row = one content page")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

# Sketch of what a target column would look like (illustrative, not the final model)
df["needs_review_score"] = df["trend_direction"].map({
    "down": 1.0,
    "flat": 0.5,
    "new": 0.3,
    "stable": 0.2,
    "up": 0.0
})

df[["content_id", "client_id", "trend_direction", "needs_review_score"]].head(5)

Working dir: /content/YOUR_REPO_NAME
Unit of analysis: one row = one content page
30000 rows, 44 columns


,content_id,client_id,trend_direction,needs_review_score
0,content_304f48230142,client_f369cb89fc,down,1.0
1,content_a1fb4e703a9e,client_4e07408562,down,1.0
2,content_9aa793d4d895,client_7f2253d7e2,down,1.0
3,content_331d6c4de07b,client_19581e27de,stable,0.2
4,content_d99b7a2d90ca,client_3fdba35f04,down,1.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule:** In the Week 1 pipeline, a hand-written rule (a simple
threshold-based baseline) achieved 0.24 Precision@50, while a random forest trained
on the same data achieved 0.74 — roughly 3x better at flagging genuinely declining
pages in the top 50.

This gap exists because "which pages need review" isn't determined by any single
column crossing a threshold — it's a combination of many weakly-informative signals
(position tier, CTR, engagement rate, content age, competition level, etc.)
interacting together. A fixed rule can only check a small number of conditions by
hand; a model can learn the actual weighted combination of dozens of signals from
data, including interactions a human wouldn't think to hand-code (e.g. CTR mattering
differently depending on position tier). That's exactly the kind of pattern ML is
suited for and fixed rules aren't.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.